# Лабораторная работа№7,9


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix,
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from category_encoders import TargetEncoder
from textblob import TextBlob

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
df_clean_original = pd.read_csv("dataSet.csv", sep=",")
print(df_clean_original.head())
print("\nИнформация о данных:")
df_clean_original.info()
print(df_clean_original.describe())
print(f"Количетсво пропусков\n{df_clean_original.isnull().sum()}")
print(f"Количество дубликатов: {df_clean_original.duplicated().sum()}")
print(
    df_clean_original[
        "performance_category"
    ].value_counts()
)
sns.countplot(x="performance_category", data=df_clean_original)
plt.title("Распределение целевой переменной")
plt.show()

plt.figure(figsize=(8, 5))
sns.histplot(df_clean_original["final_exam_score"], kde=True)
plt.title("Распределение итоговых оценок")
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(x=df_clean_original["study_hours_per_day"])
plt.title("Boxplot study_hours_per_day")
plt.show()

numeric_cols = df_clean_original.select_dtypes(include=["int64", "float64"]).columns
plt.figure(figsize=(18, 10))
corr = df_clean_original[numeric_cols].corr()
sns.heatmap(corr, cmap="coolwarm")
plt.title("Корреляционная матрица")
plt.show()

In [ ]:
def contaminate_data(df: pd.DataFrame) -> pd.DataFrame:
    df_dirty = df.copy()
    np.random.seed(RANDOM_STATE)
    for col in df_dirty.columns:
        if col != "performance_category":
            mask = np.random.random(len(df_dirty)) < 0.3
            df_dirty.loc[mask, col] = np.nan
    dup_rows = df_dirty.sample(frac=0.05, random_state=RANDOM_STATE)
    df_dirty = pd.concat([df_dirty, dup_rows], ignore_index=True)
    df_dirty = df_dirty.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    numeric_cols = df_dirty.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        outlier_idx = np.random.choice(
            df_dirty.index, size=int(0.05 * len(df_dirty)), replace=False
        )
        df_dirty.loc[outlier_idx, col] = df_dirty[col].max() * np.random.uniform(
            2, 5, size=len(outlier_idx)
        )
    cat_cols = df_dirty.select_dtypes(include=["object"]).columns
    mistakes = {
        "Engineering": ["enginering", "engeniring"],
        "Arts": ["art"],
        "Medical": ["medicl", "medcal"],
        "Low": ["lOw", "lo"],
        "High": ["hgh", "hig", "hi"],
    }
    for col in cat_cols:
        for correct, wrong_list in mistakes.items():
            for wrong in wrong_list:
                df_dirty[col] = df_dirty[col].replace(wrong, correct)
        rows_perturb = np.random.choice(
            df_dirty.index, size=int(0.1 * len(df_dirty)), replace=False
        )
        for idx in rows_perturb:
            if (
                isinstance(df_dirty.loc[idx, col], str)
                and len(df_dirty.loc[idx, col]) > 2
            ):
                s = list(df_dirty.loc[idx, col])
                pos1, pos2 = np.random.choice(len(s), 2, replace=False)
                s[pos1], s[pos2] = s[pos2], s[pos1]
                df_dirty.loc[idx, col] = "".join(s)
    comments = [
        "Не занемается и плохо учится",
        "Абсолютно не заинтересован в учёбе",
        "Много прогуливал",
        "Попал в универ только потомучто папа знаект декана",
    ]
    df_dirty["teacher_comment"] = np.random.choice(comments, size=len(df_dirty))
    return df_dirty


df_dirty = contaminate_data(df_clean_original)
df_dirty.to_csv("dataSetDirty.csv", index=False)
print(f"Загрязнённый датасет сохранён, размер: {df_dirty.shape}")

df = pd.read_csv("dataSetDirty.csv")
print("Первые 5 строк:")
df.head()
print("\nИнформация о данных:")
df.info()
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False).plot(kind="bar", figsize=(12, 4))
plt.title("Количество пропущенных значений по признакам")
plt.show()
print(f"Дубликатов строк: {df.duplicated().sum()}")
sns.countplot(data=df, x="performance_category")
plt.title("Распределение целевой переменной (до очистки)")
plt.show()